<a href="https://colab.research.google.com/github/AdriBarrios96/Challenge2-DataScience-TelecomX/blob/main/Challenge2_TelecomX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROYECTO: Telecom X - Análisis de Churn


##📌 Extracción(E - Extract)

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
sns.set_theme(style="whitegrid")

In [6]:
datos = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/refs/heads/main/TelecomX_Data.json"

In [7]:
df = pd.read_json(datos)
df.head()

,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


##🔧 Transformación (T - Transform)

In [9]:
#vemos columnas, nulos y tipos de datos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerID  7267 non-null   object
 1   Churn       7267 non-null   object
 2   customer    7267 non-null   object
 3   phone       7267 non-null   object
 4   internet    7267 non-null   object
 5   account     7267 non-null   object
dtypes: object(6)
memory usage: 340.8+ KB


####Normalizamos el DataFrame

In [10]:
#Nomralizamos cada columna
df_customer = pd.json_normalize(df['customer'])
df_phone = pd.json_normalize(df['phone'])
df_internet = pd.json_normalize(df['internet'])
df_account = pd.json_normalize(df['account'])
#unimos toda la informacion en df_normalizado
df_normalizado = pd.concat([df[['customerID', 'Churn']], df_customer, df_phone, df_internet, df_account], axis=1)
#Visualizamos los datos
df_normalizado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7267 non-null   object 
 1   Churn             7267 non-null   object 
 2   gender            7267 non-null   object 
 3   SeniorCitizen     7267 non-null   int64  
 4   Partner           7267 non-null   object 
 5   Dependents        7267 non-null   object 
 6   tenure            7267 non-null   int64  
 7   PhoneService      7267 non-null   object 
 8   MultipleLines     7267 non-null   object 
 9   InternetService   7267 non-null   object 
 10  OnlineSecurity    7267 non-null   object 
 11  OnlineBackup      7267 non-null   object 
 12  DeviceProtection  7267 non-null   object 
 13  TechSupport       7267 non-null   object 
 14  StreamingTV       7267 non-null   object 
 15  StreamingMovies   7267 non-null   object 
 16  Contract          7267 non-null   object 


####Correxiones de formatos y limpieza

In [11]:
#Transformamos la columna a valores numericos
df_normalizado['Charges.Total'] = pd.to_numeric(df_normalizado['Charges.Total'], errors='coerce')

#Verificamos los cambios
print("Tipo de dato ahora:", df_normalizado['Charges.Total'].dtype)

#Contamos espacios en blanco (NaN)
nulos_totales = df_normalizado['Charges.Total'].isnull().sum()
print("Cantidad de valores nulos encontrados:", nulos_totales)

Tipo de dato ahora: float64
Cantidad de valores nulos encontrados: 11


In [12]:
#Volvemos 0 los NAN
df_normalizado['Charges.Total'] = df_normalizado['Charges.Total'].fillna(0)

In [13]:
#Buscamos y eliminamos filas duplicadas
duplicados = df_normalizado.duplicated().sum()
df_normalizado = df_normalizado.drop_duplicates()

In [14]:
#Creamos CUENTAS_DIARIAS
df_normalizado['Cuentas_Diarias'] = round(df_normalizado['Charges.Monthly'] / 30, 2) #dividimos en 30 porque es mensual

#Visualizamos el DF
df_normalizado[['customerID', 'Charges.Monthly', 'Cuentas_Diarias', 'Charges.Total']].head()

,customerID,Charges.Monthly,Cuentas_Diarias,Charges.Total
0,0002-ORFBO,65.6,2.19,593.30
1,0003-MKNFE,59.9,2.00,542.40
2,0004-TLHLJ,73.9,2.46,280.85
3,0011-IGKFF,98.0,3.27,1237.85
4,0013-EXCHZ,83.9,2.80,267.40


##📊 Carga y análisis(L - Load & Analysis)